# Elevator simulation

A simulation of a multiple elevators operating efficiently in a tall busy building

In [152]:
# parameteric inputs
nFloors = 50
nPassengerIds = 100
capacityPerElevator = 8
nElevators = 2

# program params
input_data_fn = "input.csv"
elevator_position_history_fn = "out/elevator_positions_log.csv"
summary_stats_fn = "out/passenger_summary_stats.csv"

nTimeSteps = 0 # will be set based on input data length


In [153]:
# imports
import pandas as pd

# built-ins
from typing import List, Literal, TypedDict

In [154]:
# load input data
input_data = pd.read_csv(input_data_fn)

nTimeSteps = int(input_data.iloc[-1, 0])
print("number of time steps:", nTimeSteps)

# print(input_data)


number of time steps: 999


In [155]:
# validate input parameters
assert nFloors > 0, "Number of floors must be positive"
assert nPassengerIds > 0, "Number of passenger IDs must be positive"
assert capacityPerElevator > 0, "Capacity per elevator must be positive"
assert nElevators > 0, "Number of elevators must be positive"
assert nTimeSteps > 0, "Number of time steps must be positive"

In [156]:
# initialise the elevator position output data structure
elevator_position_history = pd.DataFrame({
    "time": range(nTimeSteps),
    **{f"elevator{i}": [0] * nTimeSteps for i in range(nElevators)}, # assumption: all elevators start on floor 0
}, dtype=int)

elevator_position_history.to_csv(elevator_position_history_fn, index=False)

In [ ]:
# a typesafe dict data structure for keeping track of active elevator requests
class ElevatorRequest(TypedDict):
    time: int
    passenger_id: str
    source: int
    dest: int

active_requests: dict[str, ElevatorRequest] = {}

# a data structure for keeping track of elevator states
ElevatorStatus = Literal["WAITING", "RISING", "FALLING"]
class Elevator(TypedDict):
    status: ElevatorStatus
    floor: int

elevatorStates: list[list[Elevator]] = []


In [ ]:
### strategies

# this strategy does nothing and always returns floor 0
strategy_nop = lambda:[Elevator(floor=0, status="STOPPED") for _ in range(nElevators)]  # no-op strategy: always return floor 0, stateus STOPPED


# this strategy can see all active requests and the t-1 history of all elevator positions
def strategy_scan(
    currentElevatorStates: list[Elevator],
    active_requests: dict[str, ElevatorRequest],
) -> list[Elevator]:
    """Return the next state for each elevator (currently unchanged)."""

    for _, request in active_requests.items():
        if request["status"] != "waiting":
            continue
        
        # find the closest elevator to the source floor
        closest_elevator = min(range(nElevators), key=lambda i: abs(currentElevatorStates[i]["floor"] - request["source"]))
        # assign the request to the closest elevator
        request["elevator"] = closest_elevator
        request["status"] = "travelling"

    return currentElevatorStates

In [159]:
currentElevatorStates = [Elevator]

# loop through the input data and position the elevators efficiently
for t in range(nTimeSteps):
    # is there a request in this epoch?
    if t in input_data["time"].values:
        requests_at_t = input_data[input_data["time"] == t]
        for _, row in requests_at_t.iterrows():
            passenger_id = row["passenger_id"]
            active_requests[passenger_id] = { # any existing request for a passengerwill be overwritten
                "time": row["time"],
                "source": row["source"],
                "dest": row["dest"],
                "status": "waiting",
                "elevator": None,
            }
    
    # # Always update elevator positions for time t
    # for elevator in range(nElevators):
    
    # no-op strategy: keep elevators at floor 0 for now
    # newElevatorStates = strategy_nop()

    # naive scan
    currentElevatorStates = strategy_scan(currentElevatorStates, active_requests)
    # print(currentElevatorStates)

    # set the elevator position history
    elevatorStates.insert(t, currentElevatorStates)

    # prev = new
    # currentElevatorStates[elevator] = newElevatorStates

elevator_position_history = pd.DataFrame(
    [[elevator["floor"] for elevator in states] for states in elevatorStates],
    columns=[f"elevator_{i}" for i in range(nElevators)]
)
elevator_position_history.to_csv(elevator_position_history_fn, index=False)
print(elevator_position_history)

# print(currentElevatorStates)

AttributeError: 'dict_items' object has no attribute 'filter'